In [7]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml
from sklearn.metrics import accuracy_score
import time

import os
import joblib

CACHE_PATH = "mnist_784.joblib"

if os.path.exists(CACHE_PATH):
    print("📂 Loading MNIST from cache...")
    X, Y = joblib.load(CACHE_PATH)
else:
    print("🔄 Downloading the MNIST dataset... (this might take a few seconds)")
    mnist = fetch_openml('mnist_784', version=1, as_frame=False, parser='auto')
    X, Y = mnist.data, mnist.target
    joblib.dump((X, y), CACHE_PATH)
    print("💾 Cached to", CACHE_PATH)

print("✅ Ready!\n")

Y = Y.astype(int)

X = X.astype(np.float64) / 255.0

📂 Loading MNIST from cache...
✅ Ready!



In [ ]:
# bad bad bad takes way too long

class Neuron:
    def __init__(self, inputs):
        self.w = np.empty(inputs.size, dtype="float64")
        self.b = 0
        self.inputs = inputs
        self.last_output = 0

    def change_weights(self, dw):
        self.w += dw[:-1]
        self.b += dw[-1]

    def get_last_raw_output(self):
        return self.last_output
    
    def get_output(self, raw):
        return max(0, raw)
    
    def get_last_output(self):
        return self.get_output(self.get_last_raw_output())

    def calculate_raw_output(self):
        last_output = sum([self.inputs[i].get_last_output() * self.w[i] for i in range(len(self.inputs))]) + self.b

class InputNeuron(Neuron):
    def __init__(self, input):
        self.last_output = input
    
    def set_input(self, input):
        self.last_output = input

    def change_weights(self, dw):
        pass
    
    def calculate_raw_output(self):
        pass

class NeuralNetwork:
    def __init__(self, layer_sizes):
        self.layer_sizes = layer_sizes
        self.neurons = []
        self.neurons.append(np.array([InputNeuron(0) for _ in range(layer_sizes[0])]))
        for i in range(1, len(layer_sizes)):
            self.neurons.append(np.array([Neuron(self.neurons[i - 1]) for _ in range(layer_sizes[i])]))
        
    def clone(self):
        new_network = NeuralNetwork(self.layer_sizes)
        for i in range(1, len(self.layer_sizes)):
            for j in range(self.layer_sizes[i]):
                new_network.neurons[i][j].w = np.copy(self.neurons[i][j].w)
                new_network.neurons[i][j].b = self.neurons[i][j].b
        return new_network
    
    def calculate(self, inputs):
        for neuron, value in zip(self.neurons[0], inputs):
            neuron.set_input(value)
        for i in range(1, len(self.layer_sizes)):
            for neuron in self.neurons[i]:
                neuron.calculate_raw_output()
        return np.array([n.get_last_output() for n in self.neurons[-1]])
    
    def random_change(self):
        maxAllowed = 0.01
        for layer_index in range(len(self.neurons)):
            rand = rng.uniform(low=-maxAllowed, high=maxAllowed, size=(self.neurons[layer_index].size, self.neurons[layer_index][0].w.size + 1))
            for i in range(rand.shape[0]):
                self.neurons[layer_index][i].change_weights(rand[i])

In [3]:
class NeuralNetwork:
    def __init__(self, layer_sizes):
        self.layer_sizes = layer_sizes
        self.weights = []
        self.biases = []
        for i in range(1, len(layer_sizes)):
            self.weights.append(np.zeros((layer_sizes[i], layer_sizes[i-1]), dtype="float64"))
            self.biases.append(np.zeros(layer_sizes[i], dtype="float64"))
        
        
    def clone(self):
        new = NeuralNetwork(self.layer_sizes)
        new.weights = [np.copy(w) for w in self.weights]
        new.biases = [np.copy(b) for b in self.biases]
        return new
    
    def calculate(self, inputs):
        curr = inputs
        for i in range(len(self.weights)):
            curr = self.weights[i]@curr + self.biases[i][:, None]
        return curr
        
    
    def random_change(self):
        maxAllowed = 0.01
        for i in range(len(self.weights)):
            self.weights[i] += rng.uniform(low=-maxAllowed, high=maxAllowed, size=self.weights[i].shape)
            self.biases[i] += rng.uniform(low=-maxAllowed, high=maxAllowed, size=self.biases[i].shape)

In [14]:
variations = 50
loops = 600

In [15]:
best_model = NeuralNetwork([28*28, 16, 10])
total_time_model_trained = 0

In [5]:
import json

with open('neural_network_mnist_random_changes.json', 'r') as file:
    data = json.load(file)

total_time_model_trained = data["total_training_time"]

best_model = NeuralNetwork(data["layer_sizes"])
for i in range(len(data["weights"])):
    best_model.weights[i] = np.array(data["weights"][i])
    best_model.biases[i] = np.array(data["biases"][i])

print(best_model.layer_sizes)

[784, 16, 10]


In [16]:
import sys
import time
def print_progress(current_iteration, max_iterations, display_eta = True, eta_update_rate = 3):
    if not hasattr(print_progress, "start_seconds"):
        print_progress.start_seconds = time.time() # Initialize on first call
    elif current_iteration == 0:
        print_progress.start_seconds = time.time()
    
    if not hasattr(print_progress, "last_time_seconds"):
        print_progress.last_time_seconds = time.time() - 0.000001
    elif current_iteration == 0:
        print_progress.last_time_seconds = time.time() - 0.000001
    

    
    progress = current_iteration/max_iterations
    sys.stdout.write("\r")
    sys.stdout.write(str(int(100*progress)) + "%")
    if display_eta and current_iteration % eta_update_rate == 1:
        sys.stdout.write(" Estimated total time: " + str(((1/progress)*(time.time()-print_progress.start_seconds))) + " seconds")
    sys.stdout.flush()

    print_progress.last_time_seconds = time.time()




def get_predictions(model, X):
    outputs = model.calculate(X.T)          # shape (10, n_samples)
    return np.argmax(outputs, axis=0)       # shape (n_samples,)


new_best_model = best_model
best_accuracy = accuracy_score(Y, get_predictions(new_best_model, X))
starting_accuracy = best_accuracy
new_best_accuracy = best_accuracy

training_start_time = time.time()

for i in range(loops):
    for _ in range(variations):

        new_model = best_model.clone()
        new_model.random_change()

        accuracy = accuracy_score(Y, get_predictions(new_model, X))
        if accuracy > new_best_accuracy:
            new_best_model = new_model
            new_best_accuracy = accuracy
    best_model = new_best_model
    best_accuracy = new_best_accuracy
    print_progress(i, loops)

training_end_time = time.time()

total_time_model_trained += (training_end_time - training_start_time)

print("\n\n")
print(best_accuracy)
print(starting_accuracy)
print(total_time_model_trained)

99% Estimated total time: 5209.642408125376 secondss


0.7889714285714285
0.09861428571428571
5208.528057813644


In [ ]:
import json

data = {
    "total_training_time": total_time_model_trained,
    "layer_sizes": best_model.layer_sizes,
    "weights": [best_model.weights[i].tolist() for i in range(len(best_model.weights))],
    "biases": [best_model.biases[i].tolist() for i in range(len(best_model.biases))]
}

with open("neural_network_mnist_random_changes.json", "w") as file:
    json.dump(data, file)